In [ ]:
"""
v10_road_damage_pipeline.py
YOLO11 Road Damage Detection — v10 (single split, ไม่มี k-fold)

แหล่งข้อมูล:
  images : data/data_crack_filter/images
  labels : data/data_crack_filter/labels

ขั้นตอน:
  1) Preprocess : ครอปแดชบอร์ด GoPro + รวมกล่องซ้ำ  -> data_clean_v10/
  2) Grouping   : จัดกลุ่มภาพตามเวลา/เฟรม (แยกตามอุปกรณ์) กันข้อมูลรั่วข้าม split
  3) Split      : stratify ตามจำนวน crack + สมดุลอุปกรณ์ -> dataset_v10/
  4) Train/Eval : เทรน YOLO11 แล้วประเมินด้วยโปรโตคอลคงที่ (conf=0.001, iou=0.6, no TTA)

วิธีรัน: รันจากโฟลเดอร์ Road_Detection/
  python v10_road_damage_pipeline.py
"""
from __future__ import annotations

import gc
import logging
import os
import random
import re
import shutil
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import NamedTuple, Sequence

import cv2
import numpy as np
import torch
import yaml
from ultralytics import YOLO

logger = logging.getLogger("v10")

IMAGE_EXTS = frozenset({".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"})
SPLIT_NAMES = ("train", "val", "test")


# ==================== 0) CONFIG ====================
@dataclass(frozen=True)
class PipelineConfig:
    base_dir: Path = field(default_factory=Path.cwd)
    seed: int = 0
    project_name: str = "road-damage-v10"
    run_name: str = "v10"
    class_names: tuple[str, ...] = ("pothole", "crack", "manhole")
    crack_id: int = 1

    # --- preprocess ---
    dashboard_crop_ratio: float = 0.15   # ครอปด้านล่างภาพ GoPro ออก 15%
    min_box_keep_fraction: float = 0.6   # กล่องเหลือ < 60% หลังครอป -> ทิ้ง
    dup_iou_merge: float = 0.70          # กล่องคลาสเดียวกัน IoU >= ค่านี้ -> รวม

    # --- grouping / split ---
    time_gap_sec: int = 90
    frame_gap: int = 5
    max_group_size: int = 25
    split_ratios: tuple[float, float, float] = (0.75, 0.15, 0.10)  # train / val / test
    crack_oversample_ratio: float = 1.5

    # --- โปรโตคอลประเมินผล: ห้ามเปลี่ยนระหว่างเวอร์ชัน ---
    val_conf: float = 0.001
    val_iou: float = 0.6

    # --- แหล่งข้อมูลต้นทาง ---
    @property
    def src_dir(self) -> Path:
        return self.base_dir / "data" / "data_crack_filter"

    @property
    def src_image_dir(self) -> Path:
        return self.src_dir / "images"

    @property
    def src_label_dir(self) -> Path:
        return self.src_dir / "labels"

    # --- โฟลเดอร์ผลลัพธ์ (สร้างใต้ base_dir) ---
    @property
    def clean_dir(self) -> Path:
        return self.base_dir / "data_clean_v10"

    @property
    def work_dir(self) -> Path:
        return self.base_dir / "dataset_v10"


MODEL_WEIGHTS = "yolo11s.pt"

# ส่งเข้า model.train() ตรง ๆ (ชื่อ key ต้องตรงกับ argument ของ ultralytics)
TRAIN_HPARAMS: dict[str, object] = {
    "imgsz": 1280,
    "batch": 8,
    "workers": min(24, os.cpu_count() or 1),
    "epochs": 200,
    "patience": 35,
    "optimizer": "AdamW",
    "lr0": 0.0015,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "dropout": 0.0,
    "cos_lr": False,
    "box": 7.5,
    "cls": 0.6,
    "dfl": 1.5,
    "mosaic": 1.0,
    "close_mosaic": 15,
    "mixup": 0.05,
    "copy_paste": 0.15,
    "degrees": 2.0,
    "translate": 0.1,
    "scale": 0.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
}

# ผล test set ของเวอร์ชันก่อน (conf=0.001, iou=0.6, no TTA): (mAP50, mAP50-95)
# หมายเหตุ: v4-v8 ใช้ข้อมูลชุด data/images เดิม -> ใช้เป็นค่าอ้างอิงเท่านั้น
PREVIOUS_RESULTS: dict[str, tuple[float, float]] = {
    "v4": (0.376, 0.149),
    "v5": (0.432, 0.174),
    "v6": (0.434, 0.170),
    "v7": (0.450, 0.178),
    "v8": (0.446, 0.171),
}


class Box(NamedTuple):
    """กล่อง YOLO แบบ normalized 0-1"""
    cls: int
    xc: float
    yc: float
    w: float
    h: float


@dataclass(frozen=True)
class EvalResult:
    map50: float
    map50_95: float
    precision: float
    recall: float


# ==================== 1) SYSTEM UTILITIES ====================
def get_device() -> int | str:
    if torch.cuda.is_available():
        logger.info("พบ GPU: %s -> ใช้ device=0", torch.cuda.get_device_name(0))
        return 0
    logger.warning("ไม่พบ GPU (CUDA) -> ใช้ CPU (เทรนช้ากว่ามาก)")
    return "cpu"


def setup_reproducibility(seed: int, has_gpu: bool) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    cv2.setNumThreads(0)  # กัน OpenCV แย่ง core กับ PyTorch DataLoader
    if has_gpu:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def free_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_rmtree(path: Path, root: Path) -> None:
    """ลบโฟลเดอร์ได้เฉพาะเมื่ออยู่ใต้ root เท่านั้น (กันลบผิดที่)"""
    path, root = path.resolve(), root.resolve()
    if path == root or root not in path.parents:
        raise ValueError(f"ปฏิเสธการลบโฟลเดอร์นอก {root}: {path}")
    if path.exists():
        shutil.rmtree(path)


def list_images(img_dir: Path) -> list[Path]:
    if not img_dir.is_dir():
        raise FileNotFoundError(f"ไม่พบโฟลเดอร์ภาพ: {img_dir}")
    return sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)


# ==================== 2) FILENAME PARSING ====================
_VLC_TS_RE = re.compile(
    r"^vlcsnap[_-](\d{4})-(\d{2})-(\d{2})-(\d{2})h(\d{2})m(\d{2})s\d*$", re.IGNORECASE
)
_PHONE_TS_RE = re.compile(r"^(\d{8})_(\d{6})$")
_FRAME_RE = re.compile(r"^([A-Za-z]+)-(\d+)(?:-[A-Za-z0-9]+)?$")


def extract_timestamp(filename: str) -> datetime | None:
    """ดึงเวลาจากชื่อไฟล์ (VLC screenshot ของ GoPro หรือภาพมือถือ YYYYMMDD_HHMMSS)"""
    stem = Path(filename).stem
    try:
        if m := _VLC_TS_RE.match(stem):
            return datetime(*map(int, m.groups()))
        if m := _PHONE_TS_RE.match(stem):
            return datetime.strptime("".join(m.groups()), "%Y%m%d%H%M%S")
    except ValueError:
        logger.warning("วันที่ในชื่อไฟล์ไม่ถูกต้อง: %s", filename)
    return None


def extract_frame_number(filename: str) -> tuple[str, int] | None:
    """ดึง (prefix, เลขเฟรม) เช่น 'vlcsnap-00001' -> ('vlcsnap', 1)"""
    if m := _FRAME_RE.match(Path(filename).stem):
        return m.group(1), int(m.group(2))
    return None


def get_device_type(filename: str) -> str:
    """'gopro' ถ้าชื่อขึ้นต้นด้วย vlcsnap, นอกนั้นถือเป็น 'phone' (ปลอดภัย: ไม่ถูกครอป)"""
    return "gopro" if Path(filename).stem.lower().startswith("vlcsnap") else "phone"


# ==================== 3) LABEL I/O ====================
def read_yolo_labels(label_path: Path) -> list[Box]:
    if not label_path.exists():
        return []
    boxes: list[Box] = []
    with label_path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            parts = line.split()
            if len(parts) != 5:
                continue
            try:
                cls = int(float(parts[0]))
                xc, yc, w, h = map(float, parts[1:])
            except ValueError:
                logger.warning("label เสีย %s:%d -> ข้าม", label_path.name, line_no)
                continue
            boxes.append(Box(cls, xc, yc, w, h))
    return boxes


def write_yolo_labels(label_path: Path, boxes: Sequence[Box]) -> None:
    with label_path.open("w", encoding="utf-8") as f:
        for b in boxes:
            f.write(f"{b.cls} {b.xc:.6f} {b.yc:.6f} {b.w:.6f} {b.h:.6f}\n")


# ==================== 4) PREPROCESS: ครอปแดชบอร์ด + รวมกล่องซ้ำ ====================
def crop_dashboard_labels(boxes: Sequence[Box], crop_ratio: float,
                          min_keep_fraction: float) -> list[Box]:
    """ปรับพิกัดกล่องหลังครอปขอบล่างออก crop_ratio (ขอบบน y=0 ไม่เปลี่ยน)"""
    keep_frac = 1.0 - crop_ratio
    out: list[Box] = []
    for b in boxes:
        new_top = max(0.0, b.yc - b.h / 2)
        new_bot = min(keep_frac, b.yc + b.h / 2)
        remaining = new_bot - new_top
        if remaining <= 1e-4 or remaining < min_keep_fraction * b.h:
            continue  # อยู่ในโซนแดชบอร์ด หรือถูกครอปจนไม่สมบูรณ์
        out.append(Box(b.cls, b.xc, (new_top + new_bot) / 2 / keep_frac, b.w, remaining / keep_frac))
    return out


def crop_dashboard_image(img: np.ndarray, crop_ratio: float) -> np.ndarray:
    new_h = max(1, int(img.shape[0] * (1.0 - crop_ratio)))
    return img[:new_h]  # รองรับทั้งภาพ 2D (gray) และ 3D (color)


def iou_xywh(a: Box, b: Box) -> float:
    ix1 = max(a.xc - a.w / 2, b.xc - b.w / 2)
    iy1 = max(a.yc - a.h / 2, b.yc - b.h / 2)
    ix2 = min(a.xc + a.w / 2, b.xc + b.w / 2)
    iy2 = min(a.yc + a.h / 2, b.yc + b.h / 2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    union = a.w * a.h + b.w * b.h - inter
    return inter / union if union > 0 else 0.0


def merge_duplicate_boxes(boxes: Sequence[Box], iou_thresh: float) -> list[Box]:
    """
    รวมกล่องคลาสเดียวกันที่ IoU >= iou_thresh กับกล่อง seed ให้เป็น union box เดียว
    (พฤติกรรมเหมือน v9 เดิม: เทียบกับ seed เท่านั้น ไม่ใช่ transitive clustering)
    """
    by_cls: dict[int, list[Box]] = defaultdict(list)
    for b in boxes:
        by_cls[b.cls].append(b)

    merged: list[Box] = []
    for cls, cls_boxes in by_cls.items():
        used = [False] * len(cls_boxes)
        for i, seed_box in enumerate(cls_boxes):
            if used[i]:
                continue
            used[i] = True
            cluster = [seed_box]
            for j in range(i + 1, len(cls_boxes)):
                if not used[j] and iou_xywh(seed_box, cls_boxes[j]) >= iou_thresh:
                    cluster.append(cls_boxes[j])
                    used[j] = True
            x1 = min(b.xc - b.w / 2 for b in cluster)
            x2 = max(b.xc + b.w / 2 for b in cluster)
            y1 = min(b.yc - b.h / 2 for b in cluster)
            y2 = max(b.yc + b.h / 2 for b in cluster)
            merged.append(Box(cls, (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
    return merged


def preprocess_dataset(cfg: PipelineConfig) -> dict[str, str]:
    """ครอป GoPro + รวมกล่องซ้ำ -> clean_dir. คืนค่า {ชื่อไฟล์ภาพ: device}"""
    # ตรวจแหล่งข้อมูลก่อน แล้วค่อยลบ clean_dir เดิม (path ผิดจะไม่ทำให้ข้อมูลเก่าหาย)
    src_label_dir = cfg.src_label_dir
    if not src_label_dir.is_dir():
        raise FileNotFoundError(f"ไม่พบโฟลเดอร์ label: {src_label_dir}")
    src_images = list_images(cfg.src_image_dir)  # มี check โฟลเดอร์ภาพในตัวแล้ว
    logger.info("แหล่งข้อมูล: images=%s | labels=%s", cfg.src_image_dir, src_label_dir)

    img_dst = cfg.clean_dir / "images"
    lbl_dst = cfg.clean_dir / "labels"
    safe_rmtree(cfg.clean_dir, cfg.base_dir)
    img_dst.mkdir(parents=True)
    lbl_dst.mkdir(parents=True)

    device_map: dict[str, str] = {}
    seen_stems: set[str] = set()
    stats: Counter[str] = Counter()

    for img_path in src_images:
        if img_path.stem in seen_stems:
            logger.warning("ชื่อไฟล์ซ้ำ (stem เดียวกัน) ข้าม: %s", img_path.name)
            stats["dup_stem"] += 1
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            logger.warning("อ่านภาพไม่ได้ ข้าม: %s", img_path.name)
            stats["unreadable"] += 1
            continue
        seen_stems.add(img_path.stem)

        label_path = src_label_dir / f"{img_path.stem}.txt"
        if not label_path.exists():
            stats["no_label"] += 1  # ถือเป็นภาพ background (ไม่มีกล่อง)
        boxes = read_yolo_labels(label_path)
        device = get_device_type(img_path.name)
        out_img = img_dst / img_path.name

        if device == "gopro":
            n_before = len(boxes)
            boxes = crop_dashboard_labels(boxes, cfg.dashboard_crop_ratio, cfg.min_box_keep_fraction)
            stats["boxes_dropped"] += n_before - len(boxes)
            if not cv2.imwrite(str(out_img), crop_dashboard_image(img, cfg.dashboard_crop_ratio)):
                raise OSError(f"บันทึกภาพไม่สำเร็จ: {out_img}")
        else:
            shutil.copy2(img_path, out_img)  # ไม่ re-encode ภาพที่ไม่ได้แก้

        n_before = len(boxes)
        boxes = merge_duplicate_boxes(boxes, cfg.dup_iou_merge)
        stats["boxes_merged"] += n_before - len(boxes)

        write_yolo_labels(lbl_dst / f"{img_path.stem}.txt", boxes)
        device_map[img_path.name] = device
        stats[device] += 1

    logger.info("=== สรุป Preprocessing (v9) ===")
    logger.info("ภาพต้นฉบับ: %d | ใช้ได้: %d | อ่านไม่ได้: %d | ชื่อซ้ำ: %d | ไม่มี label: %d",
                len(src_images), len(device_map), stats["unreadable"],
                stats["dup_stem"], stats["no_label"])
    logger.info("GoPro (ครอปแล้ว): %d | สมาร์ทโฟน: %d", stats["gopro"], stats["phone"])
    logger.info("กล่องถูกทิ้ง (โซนแดชบอร์ด): %d | กล่องถูกรวม (ซ้ำ): %d",
                stats["boxes_dropped"], stats["boxes_merged"])
    if not device_map:
        raise RuntimeError(f"ไม่มีภาพที่ใช้ได้ใน {cfg.src_image_dir}")
    return device_map


# ==================== 5) GROUPING (กันภาพเฟรมติดกันรั่วข้าม split) ====================
def _group_one_device(paths: Sequence[Path], device: str, cfg: PipelineConfig,
                      start_gid: int) -> tuple[dict[str, list[Path]], int]:
    with_time: list[tuple[datetime, Path]] = []
    with_frame: list[tuple[str, int, Path]] = []
    others: list[Path] = []
    for p in paths:
        if (ts := extract_timestamp(p.name)) is not None:
            with_time.append((ts, p))
        elif (frame := extract_frame_number(p.name)) is not None:
            with_frame.append((*frame, p))
        else:
            others.append(p)

    groups: dict[str, list[Path]] = defaultdict(list)
    gid = start_gid

    prev_ts: datetime | None = None
    for ts, p in sorted(with_time, key=lambda x: (x[0], x[1].name)):
        if prev_ts is None or (ts - prev_ts).total_seconds() > cfg.time_gap_sec:
            gid += 1
        groups[f"{device}_time_g{gid}"].append(p)
        prev_ts = ts

    prev: tuple[str, int] | None = None
    for prefix, num, p in sorted(with_frame, key=lambda x: (x[0], x[1])):
        if prev is None or prefix != prev[0] or num - prev[1] > cfg.frame_gap:
            gid += 1
        groups[f"{device}_frame_g{gid}"].append(p)
        prev = (prefix, num)

    for p in others:
        gid += 1
        groups[f"{device}_other_g{gid}"].append(p)

    return groups, gid


def build_groups(img_paths: Sequence[Path], device_map: dict[str, str],
                 cfg: PipelineConfig) -> dict[str, list[Path]]:
    """จัดกลุ่มแยกตามอุปกรณ์ -> กลุ่มเดียวกันจะไม่ปนภาพ GoPro กับมือถือ"""
    by_device: dict[str, list[Path]] = defaultdict(list)
    for p in img_paths:
        by_device[device_map.get(p.name, "phone")].append(p)

    raw_groups: dict[str, list[Path]] = {}
    gid = 0
    for device in sorted(by_device):
        dev_groups, gid = _group_one_device(by_device[device], device, cfg, gid)
        raw_groups.update(dev_groups)

    final_groups: dict[str, list[Path]] = {}
    size = cfg.max_group_size
    for key, files in raw_groups.items():
        if len(files) <= size:
            final_groups[key] = files
        else:
            for i in range(0, len(files), size):
                final_groups[f"{key}_c{i // size}"] = files[i:i + size]
    return final_groups


def count_cracks_per_image(img_paths: Sequence[Path], label_dir: Path,
                           crack_id: int) -> dict[str, int]:
    """อ่าน label ครั้งเดียว แล้ว cache จำนวน crack ต่อภาพ"""
    return {
        p.name: sum(1 for b in read_yolo_labels(label_dir / f"{p.stem}.txt") if b.cls == crack_id)
        for p in img_paths
    }


# ==================== 6) SPLIT: stratify ตาม crack + สมดุลอุปกรณ์ ====================
def stratified_split_groups(group_keys: Sequence[str], group_cracks: dict[str, int],
                            ratios: Sequence[float], seed: int) -> list[list[str]]:
    """กระจายกลุ่มลงแต่ละ split ตาม ratios โดยเรียงกลุ่มที่มี crack มากก่อน"""
    keys = list(group_keys)
    random.Random(seed).shuffle(keys)              # tie-breaker แบบสุ่ม
    keys.sort(key=lambda g: -group_cracks[g])      # stable sort

    n = len(ratios)
    buckets: list[list[str]] = [[] for _ in range(n)]
    counts = [0] * n
    for g in keys:
        total = sum(counts) + 1
        deficit = [ratios[i] - counts[i] / total for i in range(n)]
        idx = int(np.argmax(deficit))
        buckets[idx].append(g)
        counts[idx] += 1
    return buckets


def device_balanced_split(groups: dict[str, list[Path]], device_map: dict[str, str],
                          crack_counts: dict[str, int],
                          cfg: PipelineConfig) -> dict[str, list[str]]:
    group_cracks = {g: sum(crack_counts.get(p.name, 0) for p in paths) for g, paths in groups.items()}

    keys_by_device: dict[str, list[str]] = defaultdict(list)
    for gkey, paths in groups.items():
        keys_by_device[device_map.get(paths[0].name, "phone")].append(gkey)

    result: dict[str, list[str]] = {name: [] for name in SPLIT_NAMES}
    for device in sorted(keys_by_device):
        buckets = stratified_split_groups(keys_by_device[device], group_cracks,
                                          cfg.split_ratios, cfg.seed)
        for name, bucket in zip(SPLIT_NAMES, buckets):
            result[name].extend(bucket)
        logger.info("  [%s] groups -> %s", device,
                    " ".join(f"{n}={len(b)}" for n, b in zip(SPLIT_NAMES, buckets)))
    return result


# ==================== 7) OVERSAMPLE + COPY TO WORKDIR ====================
def oversample_crack_images(train_files: Sequence[Path], crack_counts: dict[str, int],
                            target_ratio: float, seed: int) -> list[Path]:
    crack_files = [p for p in train_files if crack_counts.get(p.name, 0) > 0]
    n_extra = int(len(crack_files) * (target_ratio - 1.0))
    if not crack_files or n_extra <= 0:
        return list(train_files)

    rnd = random.Random(seed)
    extra: list[Path] = []
    while n_extra > 0:
        take = min(n_extra, len(crack_files))
        extra.extend(rnd.sample(crack_files, take))
        n_extra -= take
    return list(train_files) + extra


def copy_split_to_workdir(splits: dict[str, list[Path]], label_dir: Path,
                          cfg: PipelineConfig) -> None:
    safe_rmtree(cfg.work_dir, cfg.base_dir)
    for split_name, files in splits.items():
        img_dst = cfg.work_dir / "images" / split_name
        lbl_dst = cfg.work_dir / "labels" / split_name
        img_dst.mkdir(parents=True)
        lbl_dst.mkdir(parents=True)

        name_counter: Counter[str] = Counter()
        copied = missing = 0
        for img_p in files:
            if not img_p.exists():
                missing += 1
                continue
            name_counter[img_p.name] += 1
            occ = name_counter[img_p.name]
            dst_name = img_p.name if occ == 1 else f"{img_p.stem}_dup{occ}{img_p.suffix}"
            shutil.copy2(img_p, img_dst / dst_name)

            lbl_src = label_dir / f"{img_p.stem}.txt"
            if lbl_src.exists():
                shutil.copy2(lbl_src, lbl_dst / f"{Path(dst_name).stem}.txt")
            copied += 1
        logger.info("  [%s] copied=%d missing=%d unique=%d",
                    split_name, copied, missing, len(name_counter))


def write_data_yaml(work_dir: Path, class_names: Sequence[str]) -> Path:
    yaml_path = work_dir / "data.yaml"
    content = {
        "path": str(work_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": len(class_names),
        "names": list(class_names),
    }
    with yaml_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(content, f, sort_keys=False, allow_unicode=True)
    logger.info("สร้าง %s", yaml_path)
    return yaml_path


# ==================== 8) TRAIN / EVALUATE ====================
def train_model(data_yaml: Path, cfg: PipelineConfig, device: int | str) -> Path:
    """เทรนแล้วคืน path ของ best.pt"""
    model = YOLO(MODEL_WEIGHTS)
    model.train(
        data=str(data_yaml),
        project=cfg.project_name,
        name=cfg.run_name,
        exist_ok=True,
        device=device,
        amp=device != "cpu",
        seed=cfg.seed,
        deterministic=True,
        cache=False,
        verbose=True,
        **TRAIN_HPARAMS,
    )
    best_pt = Path(model.trainer.save_dir) / "weights" / "best.pt"
    del model
    free_memory()
    if not best_pt.exists():
        raise FileNotFoundError(f"ไม่พบ best.pt: {best_pt}")
    return best_pt


def evaluate_model(model: YOLO, data_yaml: Path, split: str,
                   cfg: PipelineConfig) -> EvalResult:
    """โปรโตคอลคงที่: conf=0.001, iou=0.6, ไม่ใช้ TTA -> เทียบข้ามเวอร์ชันได้"""
    metrics = model.val(
        data=str(data_yaml),
        split=split,
        conf=cfg.val_conf,
        iou=cfg.val_iou,
        imgsz=TRAIN_HPARAMS["imgsz"],
        verbose=True,
    )
    box = metrics.box
    result = EvalResult(float(box.map50), float(box.map), float(box.mp), float(box.mr))
    logger.info("%s  mAP50: %.4f | mAP50-95: %.4f | P: %.4f | R: %.4f",
                split.upper(), result.map50, result.map50_95, result.precision, result.recall)

    # ค่าต่อคลาสเรียงตาม ap_class_index (เฉพาะคลาสที่มีใน split นั้น)
    pos_of_class = {int(c): i for i, c in enumerate(box.ap_class_index)}
    for cls_id, name in enumerate(cfg.class_names):
        i = pos_of_class.get(cls_id)
        if i is None:
            logger.info("  %-10s  (ไม่มี instance ใน %s set)", name, split)
        else:
            logger.info("  %-10s  P=%.3f  R=%.3f  AP50=%.3f",
                        name, box.p[i], box.r[i], box.ap50[i])
    return result


def log_comparison(result: EvalResult, version: str = "v9") -> None:
    logger.info("=== เทียบกับเวอร์ชันก่อน (test set, conf=0.001, iou=0.6, no TTA) ===")
    logger.info("หมายเหตุ: v4-v8 ใช้ข้อมูลคนละชุด -> ใช้เป็นค่าอ้างอิงเท่านั้น")
    for ver, (m50, m5095) in PREVIOUS_RESULTS.items():
        logger.info("%-3s TEST mAP50: %.3f | mAP50-95: %.3f", ver, m50, m5095)
    logger.info("%-3s TEST mAP50: %.4f | mAP50-95: %.4f | P: %.4f | R: %.4f",
                version, result.map50, result.map50_95, result.precision, result.recall)


# ==================== 9) PIPELINE ====================
def run_pipeline(cfg: PipelineConfig) -> EvalResult:
    device = get_device()
    setup_reproducibility(cfg.seed, has_gpu=device != "cpu")

    # 1) preprocess
    device_map = preprocess_dataset(cfg)
    clean_label_dir = cfg.clean_dir / "labels"
    clean_images = list_images(cfg.clean_dir / "images")

    # 2) grouping
    groups = build_groups(clean_images, device_map, cfg)
    logger.info("จำนวนกลุ่มภาพ (หลัง grouping กันรั่ว): %d", len(groups))

    # 3) split
    logger.info("=== แบ่ง train/val/test (stratify crack + สมดุลอุปกรณ์) ===")
    crack_counts = count_cracks_per_image(clean_images, clean_label_dir, cfg.crack_id)
    split_keys = device_balanced_split(groups, device_map, crack_counts, cfg)
    splits = {name: [p for k in keys for p in groups[k]] for name, keys in split_keys.items()}
    logger.info("ก่อน oversample -> %s",
                " ".join(f"{n}={len(f)}" for n, f in splits.items()))

    n_before = len(splits["train"])
    splits["train"] = oversample_crack_images(splits["train"], crack_counts,
                                              cfg.crack_oversample_ratio, cfg.seed)
    logger.info("หลัง oversample crack -> train: %d -> %d", n_before, len(splits["train"]))

    logger.info("=== คัดลอกไฟล์ไป workdir ===")
    copy_split_to_workdir(splits, clean_label_dir, cfg)
    data_yaml = write_data_yaml(cfg.work_dir, cfg.class_names)

    # 4) train + evaluate
    logger.info("=== เริ่มเทรน (v9) ===")
    best_pt = train_model(data_yaml, cfg, device)
    best = YOLO(str(best_pt))

    logger.info("--- ผลบน VAL SET ---")
    evaluate_model(best, data_yaml, "val", cfg)
    logger.info("--- ผลบน TEST SET ---")
    test_result = evaluate_model(best, data_yaml, "test", cfg)

    best.export(format="onnx")
    logger.info("Export ONNX สำเร็จ: %s", best_pt.parent)

    log_comparison(test_result)
    return test_result


def main() -> None:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    logger.info("=== V9 Training Pipeline (single split) ===")
    run_pipeline(PipelineConfig())
    logger.info("v9 เสร็จสมบูรณ์")


if __name__ == "__main__":
    main()

=== V7 Training Pipeline (full, fixed) ===
1. Group-aware split (v6 logic) + stratify ตาม crack instance
2. Copy ไฟล์จริงไปยัง dataset_clean (กัน oversample ถูกทับกัน)
3. Oversample crack เฉพาะ train set
4. CLAHE preprocessing: ปิดอยู่ (ทดสอบทีละตัวแปร)
5. Train ด้วย yolo11m, imgsz=1280, cls=0.6, patience=30
6. Validate มาตรฐาน conf=0.001, iou=0.6

จำนวนไฟล์ภาพที่ scan เจอ: 2009
จำนวนกลุ่มทั้งหมด: 99
ก่อน oversample -> Train: 1466 (73.0%) | Val: 311 (15.5%) | Test: 232 (11.5%)
Crack instances -> Train: 1891 (75.1%) | Val: 382 (15.2%) | Test: 246 (9.8%)
หลัง oversample -> Train: 1988 (มีไฟล์ซ้ำโดยตั้งใจ)
=== สรุปการคัดลอกไฟล์ตาม split ===
[TRAIN] ขอ: 1988 (ไม่ซ้ำ: 1466) | คัดลอกภาพสำเร็จ: 1988 | คัดลอก label สำเร็จ: 1988 | ไฟล์ที่หาไม่เจอ: 0
[VAL] ขอ: 311 (ไม่ซ้ำ: 311) | คัดลอกภาพสำเร็จ: 311 | คัดลอก label สำเร็จ: 311 | ไฟล์ที่หาไม่เจอ: 0
[TEST] ขอ: 232 (ไม่ซ้ำ: 232) | คัดลอกภาพสำเร็จ: 232 | คัดลอก label สำเร็จ: 232 | ไฟล์ที่หาไม่เจอ: 0

อัปเดต data.yaml แล้วที่: dataset_clean/data.yaml

[W925 09:03:15.864602648 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 419430400 bytes (free: 357367808, total: 25249579008).
[W925 09:03:15.991800180 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 52428800 bytes (free: 15532032, total: 25249579008).
[W925 09:03:15.995800827 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 52428800 bytes (free: 42795008, total: 25249579008).


train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4438.6±1443.5 MB/s, size: 91.9 KB)
train: Scanning /home/ds/Desktop/Road_Detection/dataset_clean/labels/train.cache... 1988 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1988/1988 926.5Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4452.0±1766.5 MB/s, size: 81.2 KB)
val: Scanning /home/ds/Desktop/Road_Detection/dataset_clean/labels/val.cache... 311 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 311/311 186.3Mit/s 0.0s
: 0% ──────────── 0/249  9.5s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/200      14.4G      2.385      3.772      2.352         18       1280: 100% ━━━━━━━━━━━━ 497/497 10.2it/s 48.8s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 1.3it/s 15.2s<0.1s
                   all        311        820      0.197      0.222      0.141     0.0509

      Epoch    GPU_mem   box_loss   cls_loss